In [162]:
pip install pulp

Note: you may need to restart the kernel to use updated packages.


In [163]:
import pandas as pd

food_candidates = pd.read_csv("./ready_food_data_frame.csv")


In [164]:
patient_target = {
    "diet_type": "DM_HT_OBESITY",
    "energy_kcal": 1600,
    "carb_g": 240,
    "protein_g": 60,
    "fat_g": 44,
    "sodium_mg_max": 2000,
    "fiber_g_min": 25
}

In [165]:
def filter_foods_for_disease(df, diet_type):
    filtered = df.copy()

    # DM: remove sugar category if exists
    if "DM" in diet_type:
        filtered = filtered[filtered["category_code"] != "G"]

    # Hypertension: remove very high sodium foods
    if "HT" in diet_type:
        filtered = filtered[
            filtered["sodium_mg_per_portion"].isna() |
            (filtered["sodium_mg_per_portion"] <= 400)
        ]

    # Obesity: remove very high energy per portion
    if "OBESITY" in diet_type:
        filtered = filtered[
            filtered["energy_kcal_per_portion"] <= 250
        ]

    return filtered


candidate_for_patient = filter_foods_for_disease(
    food_candidates,
    patient_target["diet_type"]
)

print("Candidate foods after disease filter:", len(candidate_for_patient))

display(
    candidate_for_patient[[
        "food_code",
        "food_name",
        "category_code",
        "urt",
        "gram_per_portion",
        "energy_kcal_per_portion",
        "carb_g_per_portion",
        "sodium_mg_per_portion",
        "fiber_g_per_portion"
    ]].head(20)
)

Candidate foods after disease filter: 204


,food_code,food_name,category_code,urt,gram_per_portion,energy_kcal_per_portion,carb_g_per_portion,sodium_mg_per_portion,fiber_g_per_portion
5,AP006,"Bihun, mentah",MP,1/2 Gelas,50.0,174.00,41.050,2.50,0.600
6,AP020,"Makaroni, mentah",MP,1½ Gelas,50.0,176.50,39.350,2.50,2.450
7,AP024,Roti putih,MP,3 Iris,70.0,173.60,35.000,63.70,6.370
8,AP029,Biskuit,MP,4 Buah Besar,40.0,183.20,30.040,8.12,0.840
9,BR013,"Kentang, segar",MP,2 Buah Sedang,210.0,130.20,28.350,14.70,1.050
10,BR014,"Kentang hitam, segar",MP,12 Biji,125.0,177.50,42.125,27.50,6.750
11,CR018,"Kacang kedelai, segar",LN,2 Sendok Makan,25.0,71.50,7.525,7.00,0.725
12,CR027,"Kacang merah, segar",LN,2 Sendok Makan,25.0,42.75,7.000,1.75,0.525
13,CP005,"kacang hijau, rebus",LN,2 Sendok Makan,25.0,27.25,4.575,111.75,0.375
14,CP007,"Kacang kedelai, rebus",LN,2 Sendok Makan,25.0,47.25,3.175,45.25,0.400


In [166]:
import pandas as pd
import numpy as np
import pulp

# ============================================================
# 1. Prepare MILP input data
# ============================================================

milp_df = candidate_for_patient.copy().reset_index(drop=True)

# Required nutrient columns
required_cols = [
    "food_name",
    "category_code",
    "urt",
    "gram_per_portion",
    "energy_kcal_per_portion",
    "protein_g_per_portion",
    "fat_g_per_portion",
    "carb_g_per_portion",
    "sodium_mg_per_portion",
    "fiber_g_per_portion"
]

# Keep only rows with required values
milp_df = milp_df.dropna(subset=[
    "category_code",
    "energy_kcal_per_portion",
    "carb_g_per_portion",
    "sodium_mg_per_portion",
    "fiber_g_per_portion"
]).copy()

# Make sure numeric columns are numeric
numeric_cols = [
    "gram_per_portion",
    "energy_kcal_per_portion",
    "protein_g_per_portion",
    "fat_g_per_portion",
    "carb_g_per_portion",
    "sodium_mg_per_portion",
    "potassium_mg_per_portion",
    "calcium_mg_per_portion",
    "fiber_g_per_portion"
]

for col in numeric_cols:
    if col in milp_df.columns:
        milp_df[col] = pd.to_numeric(milp_df[col], errors="coerce").fillna(0)

print("Total MILP candidate foods:", len(milp_df))

display(
    milp_df.groupby("category_code")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

Total MILP candidate foods: 175


,category_code,count
5,S,129
0,B,17
1,LH,9
2,LN,7
4,MP,6
3,M,5
6,SS,2


In [167]:
# ============================================================
# 2. Create MILP model
# ============================================================

model = pulp.LpProblem("DM_HT_Obesity_Menu_Recommendation", pulp.LpMinimize)

# Allow half-portion:
# portion_step = 0.5 means:
# x[i] = 0 -> 0 portion
# x[i] = 1 -> 0.5 portion
# x[i] = 2 -> 1 portion
# x[i] = 3 -> 1.5 portions
# x[i] = 4 -> 2 portions

portion_step = 0.5
max_portion_per_food = 2.0

x = {
    i: pulp.LpVariable(
        f"x_{i}",
        lowBound=0,
        upBound=int(max_portion_per_food / portion_step),
        cat="Integer"
    )
    for i in milp_df.index
}

# Real portion used in calculation
portion = {
    i: x[i] * portion_step
    for i in milp_df.index
}


Formulation total of nutrion in daily menu based on combination from MILP

In [168]:
# ============================================================
# 3. Define nutrient totals
# ============================================================

total_energy = pulp.lpSum(
    portion[i] * milp_df.loc[i, "energy_kcal_per_portion"]
    for i in milp_df.index
)

total_protein = pulp.lpSum(
    portion[i] * milp_df.loc[i, "protein_g_per_portion"]
    for i in milp_df.index
)

total_fat = pulp.lpSum(
    portion[i] * milp_df.loc[i, "fat_g_per_portion"]
    for i in milp_df.index
)

total_carb = pulp.lpSum(
    portion[i] * milp_df.loc[i, "carb_g_per_portion"]
    for i in milp_df.index
)

total_sodium = pulp.lpSum(
    portion[i] * milp_df.loc[i, "sodium_mg_per_portion"]
    for i in milp_df.index
)

total_fiber = pulp.lpSum(
    portion[i] * milp_df.loc[i, "fiber_g_per_portion"]
    for i in milp_df.index
)

total_potassium = pulp.lpSum(
    portion[i] * milp_df.loc[i, "potassium_mg_per_portion"]
    for i in milp_df.index
) if "potassium_mg_per_portion" in milp_df.columns else 0

total_calcium = pulp.lpSum(
    portion[i] * milp_df.loc[i, "calcium_mg_per_portion"]
    for i in milp_df.index
) if "calcium_mg_per_portion" in milp_df.columns else 0

In [169]:
portion_plan = {
    1100: {"MP": 3, "LH": 2, "LN": 2, "S": 2, "B": 1, "SS": 0, "M": 3, "G": 1},
    1200: {"MP": 3, "LH": 2, "LN": 2, "S": 2, "B": 1, "SS": 1, "M": 3, "G": 2},
    1300: {"MP": 3, "LH": 2, "LN": 2, "S": 2, "B": 2, "SS": 1, "M": 4, "G": 2},
    1400: {"MP": 3, "LH": 3, "LN": 3, "S": 2, "B": 2, "SS": 0, "M": 3, "G": 3},
    1500: {"MP": 3, "LH": 3, "LN": 3, "S": 3, "B": 3, "SS": 0, "M": 4, "G": 3},
    1600: {"MP": 4, "LH": 3, "LN": 3, "S": 3, "B": 2, "SS": 0, "M": 4, "G": 2},
    1700: {"MP": 4, "LH": 3, "LN": 3, "S": 3, "B": 3, "SS": 1, "M": 3, "G": 2},
    1800: {"MP": 4, "LH": 3, "LN": 3, "S": 3, "B": 3, "SS": 1, "M": 4, "G": 3},
    1900: {"MP": 4, "LH": 4, "LN": 3, "S": 3, "B": 3, "SS": 1, "M": 4, "G": 3},
    2000: {"MP": 4, "LH": 3, "LN": 4, "S": 4, "B": 4, "SS": 1, "M": 4, "G": 4},
    2100: {"MP": 4, "LH": 3, "LN": 4, "S": 4, "B": 4, "SS": 1, "M": 4, "G": 4},
    2200: {"MP": 4, "LH": 3, "LN": 3, "S": 4, "B": 4, "SS": 2, "M": 5, "G": 4},
    2300: {"MP": 5, "LH": 4, "LN": 4, "S": 4, "B": 4, "SS": 0, "M": 5, "G": 4},
    2400: {"MP": 5, "LH": 4, "LN": 4, "S": 4, "B": 4, "SS": 1, "M": 5, "G": 4},
    2500: {"MP": 5.5, "LH": 4, "LN": 4, "S": 4, "B": 4, "SS": 1, "M": 5, "G": 4},
}

In [170]:
def get_disease_constraints(diet_type, energy_kcal):
    """
    diet_type options:
    HT
    HT_OBESITY
    DM
    DM_OBESITY
    DM_HT
    DM_HT_OBESITY
    """

    constraints = {
        "energy_target": energy_kcal,
        "energy_mode": "maintenance",
        "carb_min_g": None,
        "carb_max_g": None,
        "fat_min_g": None,
        "fat_max_g": None,
        "sodium_max_mg": None,
        "potassium_min_mg": None,
        "calcium_min_mg": None,
        "fiber_min_g": None,
        "vegetable_fruit_min_portions": None,
        "sugar_max_portions": None
    }

    # 1. Hipertensi tanpa obesitas
    if diet_type == "HT":
        constraints["potassium_min_mg"] = 4700
        constraints["calcium_min_mg"] = 800

    # 2. Hipertensi dengan obesitas
    elif diet_type == "HT_OBESITY":
        constraints["energy_mode"] = "deficit"
        constraints["carb_min_g"] = energy_kcal * 0.55 / 4
        constraints["carb_max_g"] = energy_kcal * 0.65 / 4
        constraints["fat_min_g"] = energy_kcal * 0.20 / 9
        constraints["fat_max_g"] = energy_kcal * 0.25 / 9
        constraints["sodium_max_mg"] = 2300
        constraints["potassium_min_mg"] = 4700
        constraints["calcium_min_mg"] = 800

    # 3. DM tanpa obesitas
    elif diet_type == "DM":
        constraints["fiber_min_g"] = 25 * (energy_kcal / 1000)
        constraints["sugar_max_portions"] = 0

    # 4. DM dengan obesitas
    elif diet_type == "DM_OBESITY":
        constraints["energy_mode"] = "deficit"
        constraints["fiber_min_g"] = 25 * (energy_kcal / 1000)
        constraints["vegetable_fruit_min_portions"] = 5
        constraints["sugar_max_portions"] = 0

    # 5. DM + Hipertensi tanpa obesitas
    elif diet_type == "DM_HT":
        constraints["fiber_min_g"] = 25 * (energy_kcal / 1000)
        constraints["sodium_max_mg"] = 2300
        constraints["vegetable_fruit_min_portions"] = 5
        constraints["sugar_max_portions"] = 0

    # 6. DM + Hipertensi + Obesitas
    elif diet_type == "DM_HT_OBESITY":
        constraints["energy_mode"] = "deficit"
        constraints["carb_min_g"] = 130
        constraints["fiber_min_g"] = 25 * (energy_kcal / 1000)
        constraints["sodium_max_mg"] = 2000
        constraints["vegetable_fruit_min_portions"] = 5
        constraints["sugar_max_portions"] = 0

    return constraints

In [171]:
# ============================================================
# 3. Pick nearest available energy level
# ============================================================

def get_nearest_energy_level(energy_kcal, available_levels):
    return min(available_levels, key=lambda x: abs(x - energy_kcal))


def adjust_portion_plan_for_disease(base_plan, diet_type):
    plan = base_plan.copy()

    # DM: sugar should be avoided or minimized
    if "DM" in diet_type:
        plan["G"] = 0

    # Obesity: reduce sugar and oil portions
    if "OBESITY" in diet_type:
        plan["G"] = 0
        if "M" in plan:
            plan["M"] = max(0, plan["M"] - 1)

    # DM obesity / DM+HT: ensure vegetable + fruit >= 5 portions
    if diet_type in ["DM_OBESITY", "DM_HT", "DM_HT_OBESITY"]:
        current_sb = plan.get("S", 0) + plan.get("B", 0)
        if current_sb < 5:
            plan["S"] = plan.get("S", 0) + (5 - current_sb)

    return plan

In [172]:
# ============================================================
# 5. Example patient target
# ============================================================

diet_type = "DM_HT_OBESITY"
energy_target = 1600

constraints = get_disease_constraints(diet_type, energy_target)

nearest_energy = get_nearest_energy_level(
    energy_target,
    list(portion_plan.keys())
)

base_plan = portion_plan[nearest_energy]
adjusted_plan = adjust_portion_plan_for_disease(base_plan, diet_type)

candidate_for_patient = filter_foods_for_disease(food_candidates, diet_type)

print("Diet type:", diet_type)
print("Energy target:", energy_target)
print("Nearest portion plan:", nearest_energy)
print("\nDisease constraints:")
print(constraints)

print("\nBase portion plan:")
print(base_plan)

print("\nAdjusted portion plan:")
print(adjusted_plan)

print("\nTotal food candidates before filter:", len(food_candidates))
print("Total food candidates after disease filter:", len(candidate_for_patient))

category_after_filter = (
    candidate_for_patient
    .groupby("category_code")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(category_after_filter)

Diet type: DM_HT_OBESITY
Energy target: 1600
Nearest portion plan: 1600

Disease constraints:
{'energy_target': 1600, 'energy_mode': 'deficit', 'carb_min_g': 130, 'carb_max_g': None, 'fat_min_g': None, 'fat_max_g': None, 'sodium_max_mg': 2000, 'potassium_min_mg': None, 'calcium_min_mg': None, 'fiber_min_g': 40.0, 'vegetable_fruit_min_portions': 5, 'sugar_max_portions': 0}

Base portion plan:
{'MP': 4, 'LH': 3, 'LN': 3, 'S': 3, 'B': 2, 'SS': 0, 'M': 4, 'G': 2}

Adjusted portion plan:
{'MP': 4, 'LH': 3, 'LN': 3, 'S': 3, 'B': 2, 'SS': 0, 'M': 3, 'G': 0}

Total food candidates before filter: 212
Total food candidates after disease filter: 204


,category_code,count
5,S,151
0,B,19
1,LH,12
2,LN,8
4,MP,6
3,M,5
6,SS,3


In [173]:
# ============================================================
# 4. Category portion constraints
# ============================================================

for category, required_portion in adjusted_plan.items():

    category_indices = milp_df[
        milp_df["category_code"] == category
    ].index.tolist()

    # If category has no food and required portion is 0, skip safely
    if len(category_indices) == 0 and required_portion == 0:
        continue

    # If category has no food but required portion > 0, this is a real problem
    if len(category_indices) == 0 and required_portion > 0:
        print(
            f"Warning: no candidate food for category {category}, "
            f"but required portion is {required_portion}"
        )
        continue

    model += (
        pulp.lpSum(portion[i] for i in category_indices) == required_portion,
        f"portion_{category}"
    )

Ensure Constrain nutrion maxiumum around 10 percent

In [174]:
# ============================================================
# 5. Disease-specific nutrient constraints
# ============================================================

energy_target = constraints["energy_target"]
energy_min = energy_target * 0.90
energy_max = energy_target * 1.10

# Energy range
model += total_energy >= energy_min, "energy_min"
model += total_energy <= energy_max, "energy_max"

# Carbohydrate minimum
if constraints.get("carb_min_g") is not None:
    model += total_carb >= constraints["carb_min_g"], "carb_min"

# Carbohydrate maximum, if available
if constraints.get("carb_max_g") is not None:
    model += total_carb <= constraints["carb_max_g"], "carb_max"

# Fat minimum, if available
if constraints.get("fat_min_g") is not None:
    model += total_fat >= constraints["fat_min_g"], "fat_min"

# Fat maximum, if available
if constraints.get("fat_max_g") is not None:
    model += total_fat <= constraints["fat_max_g"], "fat_max"

# Sodium maximum
if constraints.get("sodium_max_mg") is not None:
    model += total_sodium <= constraints["sodium_max_mg"], "sodium_max"

# Fiber minimum
if constraints.get("fiber_min_g") is not None:
    model += total_fiber >= constraints["fiber_min_g"], "fiber_min"

In [175]:
# ============================================================
# 6. Objective function: minimize energy deviation
# ============================================================

energy_dev_pos = pulp.LpVariable("energy_dev_pos", lowBound=0)
energy_dev_neg = pulp.LpVariable("energy_dev_neg", lowBound=0)

model += total_energy - energy_target == energy_dev_pos - energy_dev_neg

# Objective:
# minimize energy deviation
# plus small penalty for sodium
model += (
    energy_dev_pos
    + energy_dev_neg
    + 0.01 * total_sodium
), "minimize_energy_deviation_and_sodium"

In [176]:
# ============================================================
# 7. Solve model
# ============================================================

solver = pulp.PULP_CBC_CMD(msg=True)
model.solve(solver)

print("Solver status:", pulp.LpStatus[model.status])
print("Objective value:", pulp.value(model.objective))

Solver status: Optimal
Objective value: 1.04975


In [177]:
selected_items = []

for i in milp_df.index:
    val = pulp.value(x[i])

    if val is not None and val > 0:
        row = milp_df.loc[i].copy()
        row["selected_portions"] = val * portion_step
        selected_items.append(row)

milp_menu = pd.DataFrame(selected_items)

print("Solver status:", pulp.LpStatus[model.status])
print("Number of selected foods:", len(milp_menu))

if not milp_menu.empty:
    display(
        milp_menu[[
            "food_name",
            "category_code",
            "urt",
            "gram_per_portion",
            "selected_portions",
            "energy_kcal_per_portion",
            "protein_g_per_portion",
            "fat_g_per_portion",
            "carb_g_per_portion",
            "sodium_mg_per_portion",
            "fiber_g_per_portion"
        ]]
    )
else:
    print("milp_menu is empty. The extraction variable may not be the same variable used in the solved model.")

Solver status: Optimal
Number of selected foods: 14


,food_name,category_code,urt,gram_per_portion,selected_portions,energy_kcal_per_portion,protein_g_per_portion,fat_g_per_portion,carb_g_per_portion,sodium_mg_per_portion,fiber_g_per_portion
0,"Bihun, mentah",MP,1/2 Gelas,50.0,2.0,174.00,2.350,0.050,41.050,2.50,0.60
1,"Makaroni, mentah",MP,1½ Gelas,50.0,2.0,176.50,4.350,0.200,39.350,2.50,2.45
12,"Tahu, mentah",LN,2 Potong Sedang,100.0,1.0,80.00,10.900,4.700,0.800,2.00,0.10
13,"Tempe kedelai, mentah",LN,2 potong sedang,50.0,2.0,100.50,10.400,4.400,6.750,4.50,0.70
64,"Daun melinjo, segar",S,1 gelas,100.0,1.5,104.00,5.000,1.300,21.300,12.20,10.30
108,"Kelawi, kluwih, segar",S,1 gelas,100.0,0.5,118.00,1.500,0.300,27.200,2.00,5.00
131,"Nangka muda, segar",S,1 gelas,100.0,1.0,57.00,2.000,0.400,11.300,1.10,8.30
165,"Duku, segar",B,14 buah sedang,80.0,2.0,50.40,0.800,0.160,12.880,1.60,3.44
189,"Sapi, daging, kurus, segar",LH,1 potong sedang,35.0,0.5,60.90,6.860,3.500,0.000,29.05,0.00
190,"Belut, segar",LH,3 ekor,45.0,0.5,31.50,6.570,0.360,0.450,24.75,0.00


In [178]:
milp_menu["selected_gram"] = (
    milp_menu["gram_per_portion"] * milp_menu["selected_portions"]
)

In [179]:
nutrient_cols = [
    "energy_kcal_per_portion",
    "protein_g_per_portion",
    "fat_g_per_portion",
    "carb_g_per_portion",
    "sodium_mg_per_portion",
    "potassium_mg_per_portion",
    "calcium_mg_per_portion",
    "fiber_g_per_portion"
]

for col in nutrient_cols:
    total_col = col.replace("_per_portion", "_selected_total")
    milp_menu[total_col] = (
        milp_menu[col] * milp_menu["selected_portions"]
    )

display(
    milp_menu[[
        "food_name",
        "category_code",
        "urt",
        "gram_per_portion",
    ] ]
)

,food_name,category_code,urt,gram_per_portion
0,"Bihun, mentah",MP,1/2 Gelas,50.0
1,"Makaroni, mentah",MP,1½ Gelas,50.0
12,"Tahu, mentah",LN,2 Potong Sedang,100.0
13,"Tempe kedelai, mentah",LN,2 potong sedang,50.0
64,"Daun melinjo, segar",S,1 gelas,100.0
108,"Kelawi, kluwih, segar",S,1 gelas,100.0
131,"Nangka muda, segar",S,1 gelas,100.0
165,"Duku, segar",B,14 buah sedang,80.0
189,"Sapi, daging, kurus, segar",LH,1 potong sedang,35.0
190,"Belut, segar",LH,3 ekor,45.0


In [180]:
from fractions import Fraction
import math

def decimal_to_fraction_text(value, max_denominator=8):
    if pd.isna(value):
        return ""

    value = float(value)

    if math.isclose(value, 0):
        return "0"

    whole = int(value)
    frac = value - whole

    frac_obj = Fraction(frac).limit_denominator(max_denominator)

    if frac_obj.numerator == 0:
        return str(whole)

    if whole == 0:
        return f"{frac_obj.numerator}/{frac_obj.denominator}"

    return f"{whole} {frac_obj.numerator}/{frac_obj.denominator}"


milp_menu["selected_urt_qty"] = (
    milp_menu["urt_qty"] * milp_menu["selected_portions"]
)

milp_menu["selected_urt_fraction"] = milp_menu["selected_urt_qty"].apply(
    decimal_to_fraction_text
)

milp_menu["selected_urt"] = (
    milp_menu["selected_urt_fraction"]
    + " "
    + milp_menu["urt_unit"]
)

display(
    milp_menu[[
        "food_name",
        "category_code",
        "selected_portions",
        "gram_per_portion",
        "selected_gram",
        "urt",
        "selected_urt"
    ]]
)

,food_name,category_code,selected_portions,gram_per_portion,selected_gram,urt,selected_urt
0,"Bihun, mentah",MP,2.0,50.0,100.0,1/2 Gelas,1 Gelas
1,"Makaroni, mentah",MP,2.0,50.0,100.0,1½ Gelas,3 Gelas
12,"Tahu, mentah",LN,1.0,100.0,100.0,2 Potong Sedang,2 Potong Sedang
13,"Tempe kedelai, mentah",LN,2.0,50.0,100.0,2 potong sedang,4 potong sedang
64,"Daun melinjo, segar",S,1.5,100.0,150.0,1 gelas,1 1/2 gelas
108,"Kelawi, kluwih, segar",S,0.5,100.0,50.0,1 gelas,1/2 gelas
131,"Nangka muda, segar",S,1.0,100.0,100.0,1 gelas,1 gelas
165,"Duku, segar",B,2.0,80.0,160.0,14 buah sedang,28 buah sedang
189,"Sapi, daging, kurus, segar",LH,0.5,35.0,17.5,1 potong sedang,1/2 potong sedang
190,"Belut, segar",LH,0.5,45.0,22.5,3 ekor,1 1/2 ekor


In [181]:
total_summary = (
    milp_menu[
        [col.replace("_per_portion", "_selected_total") for col in nutrient_cols]
    ]
    .sum()
    .reset_index()
)

total_summary.columns = ["nutrient", "total"]

display(total_summary)

,nutrient,total
0,energy_kcal_selected_total,1600.000
1,protein_g_selected_total,78.230
2,fat_g_selected_total,34.225
3,carb_g_selected_total,258.025
4,sodium_mg_selected_total,104.975
5,potassium_mg_selected_total,2246.445
6,calcium_mg_selected_total,862.400
7,fiber_g_selected_total,40.730


In [182]:
total_energy_value = milp_menu["energy_kcal_selected_total"].sum()
total_protein_value = milp_menu["protein_g_selected_total"].sum()
total_fat_value = milp_menu["fat_g_selected_total"].sum()
total_carb_value = milp_menu["carb_g_selected_total"].sum()
total_sodium_value = milp_menu["sodium_mg_selected_total"].sum()
total_fiber_value = milp_menu["fiber_g_selected_total"].sum()

validation_rows = []

validation_rows.append({
    "constraint": "energy",
    "value": total_energy_value,
    "target_or_limit": f"{energy_min} - {energy_max}",
    "status": "pass" if energy_min <= total_energy_value <= energy_max else "fail"
})

if constraints.get("carb_min_g") is not None:
    validation_rows.append({
        "constraint": "carb_min",
        "value": total_carb_value,
        "target_or_limit": f">= {constraints['carb_min_g']}",
        "status": "pass" if total_carb_value >= constraints["carb_min_g"] else "fail"
    })

if constraints.get("carb_max_g") is not None:
    validation_rows.append({
        "constraint": "carb_max",
        "value": total_carb_value,
        "target_or_limit": f"<= {constraints['carb_max_g']}",
        "status": "pass" if total_carb_value <= constraints["carb_max_g"] else "fail"
    })

if constraints.get("sodium_max_mg") is not None:
    validation_rows.append({
        "constraint": "sodium_max",
        "value": total_sodium_value,
        "target_or_limit": f"<= {constraints['sodium_max_mg']}",
        "status": "pass" if total_sodium_value <= constraints["sodium_max_mg"] else "fail"
    })

if constraints.get("fiber_min_g") is not None:
    validation_rows.append({
        "constraint": "fiber_min",
        "value": total_fiber_value,
        "target_or_limit": f">= {constraints['fiber_min_g']}",
        "status": "pass" if total_fiber_value >= constraints["fiber_min_g"] else "fail"
    })

validation_df = pd.DataFrame(validation_rows)
display(validation_df)

,constraint,value,target_or_limit,status
0,energy,1600.000,1440.0 - 1760.0000000000002,pass
1,carb_min,258.025,>= 130,pass
2,sodium_max,104.975,<= 2000,pass
3,fiber_min,40.730,>= 40.0,pass


In [183]:
portion_check = (
    milp_menu
    .groupby("category_code")["selected_portions"]
    .sum()
    .reset_index()
    .rename(columns={"selected_portions": "selected_portions_total"})
)

adjusted_plan_df = pd.DataFrame([
    {"category_code": k, "required_portions": v}
    for k, v in adjusted_plan.items()
])

portion_check = adjusted_plan_df.merge(
    portion_check,
    on="category_code",
    how="left"
).fillna(0)

portion_check["status"] = np.where(
    portion_check["required_portions"] == portion_check["selected_portions_total"],
    "pass",
    "fail"
)

display(portion_check)

,category_code,required_portions,selected_portions_total,status
0,MP,4,4.0,pass
1,LH,3,3.0,pass
2,LN,3,3.0,pass
3,S,3,3.0,pass
4,B,2,2.0,pass
5,SS,0,0.0,pass
6,M,3,3.0,pass
7,G,0,0.0,pass


In [184]:
# ============================================================
# 8. Evaluate menu against constraints
# ============================================================

def evaluate_menu(daily_totals, constraints):
    evaluation = []

    energy_total = daily_totals.get("energy_kcal_per_portion", np.nan)
    carb_total = daily_totals.get("carb_g_per_portion", np.nan)
    protein_total = daily_totals.get("protein_g_per_portion", np.nan)
    fat_total = daily_totals.get("fat_g_per_portion", np.nan)
    sodium_total = daily_totals.get("sodium_mg_per_portion", np.nan)
    potassium_total = daily_totals.get("potassium_mg_per_portion", np.nan)
    calcium_total = daily_totals.get("calcium_mg_per_portion", np.nan)
    fiber_total = daily_totals.get("fiber_g_per_portion", np.nan)

    # Energy: allow ±10% for rule-based prototype
    energy_target = constraints["energy_target"]
    energy_min = energy_target * 0.90
    energy_max = energy_target * 1.10

    evaluation.append({
        "constraint": "energy",
        "value": energy_total,
        "target_or_limit": f"{energy_min:.1f} - {energy_max:.1f}",
        "status": "pass" if energy_min <= energy_total <= energy_max else "not_pass"
    })

    # Carbohydrate minimum
    if constraints.get("carb_min_g") is not None:
        evaluation.append({
            "constraint": "carb_min",
            "value": carb_total,
            "target_or_limit": f">= {constraints['carb_min_g']:.1f}",
            "status": "pass" if carb_total >= constraints["carb_min_g"] else "not_pass"
        })

    # Carbohydrate maximum
    if constraints.get("carb_max_g") is not None:
        evaluation.append({
            "constraint": "carb_max",
            "value": carb_total,
            "target_or_limit": f"<= {constraints['carb_max_g']:.1f}",
            "status": "pass" if carb_total <= constraints["carb_max_g"] else "not_pass"
        })

    # Fat minimum
    if constraints.get("fat_min_g") is not None:
        evaluation.append({
            "constraint": "fat_min",
            "value": fat_total,
            "target_or_limit": f">= {constraints['fat_min_g']:.1f}",
            "status": "pass" if fat_total >= constraints["fat_min_g"] else "not_pass"
        })

    # Fat maximum
    if constraints.get("fat_max_g") is not None:
        evaluation.append({
            "constraint": "fat_max",
            "value": fat_total,
            "target_or_limit": f"<= {constraints['fat_max_g']:.1f}",
            "status": "pass" if fat_total <= constraints["fat_max_g"] else "not_pass"
        })

    # Sodium maximum
    if constraints.get("sodium_max_mg") is not None:
        evaluation.append({
            "constraint": "sodium_max",
            "value": sodium_total,
            "target_or_limit": f"<= {constraints['sodium_max_mg']:.1f}",
            "status": "pass" if sodium_total <= constraints["sodium_max_mg"] else "not_pass"
        })

    # Fiber minimum
    if constraints.get("fiber_min_g") is not None:
        evaluation.append({
            "constraint": "fiber_min",
            "value": fiber_total,
            "target_or_limit": f">= {constraints['fiber_min_g']:.1f}",
            "status": "pass" if fiber_total >= constraints["fiber_min_g"] else "not_pass"
        })

    # Potassium minimum
    if constraints.get("potassium_min_mg") is not None:
        evaluation.append({
            "constraint": "potassium_min",
            "value": potassium_total,
            "target_or_limit": f">= {constraints['potassium_min_mg']:.1f}",
            "status": "pass" if potassium_total >= constraints["potassium_min_mg"] else "not_pass"
        })

    # Calcium minimum
    if constraints.get("calcium_min_mg") is not None:
        evaluation.append({
            "constraint": "calcium_min",
            "value": calcium_total,
            "target_or_limit": f">= {constraints['calcium_min_mg']:.1f}",
            "status": "pass" if calcium_total >= constraints["calcium_min_mg"] else "not_pass"
        })

    return pd.DataFrame(evaluation)




In [185]:
# ============================================================
# 9. Calculate MILP nutrient totals
# ============================================================

nutrient_cols = [
    "energy_kcal_per_portion",
    "protein_g_per_portion",
    "fat_g_per_portion",
    "carb_g_per_portion",
    "sodium_mg_per_portion",
    "potassium_mg_per_portion",
    "calcium_mg_per_portion",
    "fiber_g_per_portion"
]

existing_nutrient_cols = [
    col for col in nutrient_cols if col in milp_menu.columns
]

milp_totals = {}

for col in existing_nutrient_cols:
    milp_totals[col] = (
        milp_menu[col] * milp_menu["selected_portions"]
    ).sum()

milp_totals = pd.Series(milp_totals)

display(
    milp_totals
    .reset_index()
    .rename(columns={"index": "nutrient", 0: "total"})
)

,nutrient,total
0,energy_kcal_per_portion,1600.000
1,protein_g_per_portion,78.230
2,fat_g_per_portion,34.225
3,carb_g_per_portion,258.025
4,sodium_mg_per_portion,104.975
5,potassium_mg_per_portion,2246.445
6,calcium_mg_per_portion,862.400
7,fiber_g_per_portion,40.730


In [186]:
milp_evaluation = evaluate_menu(
    milp_totals,
    constraints
)

display(milp_evaluation)

,constraint,value,target_or_limit,status
0,energy,1600.000,1440.0 - 1760.0,pass
1,carb_min,258.025,>= 130.0,pass
2,sodium_max,104.975,<= 2000.0,pass
3,fiber_min,40.730,>= 40.0,pass


In [187]:
selected_category_count = (
    milp_menu
    .groupby("category_code")["selected_portions"]
    .sum()
    .reset_index(name="selected_portions")
)

display(selected_category_count)

print("Adjusted portion plan:")
print(adjusted_plan)

,category_code,selected_portions
0,B,2.0
1,LH,3.0
2,LN,3.0
3,M,3.0
4,MP,4.0
5,S,3.0


Adjusted portion plan:
{'MP': 4, 'LH': 3, 'LN': 3, 'S': 3, 'B': 2, 'SS': 0, 'M': 3, 'G': 0}


In [188]:
milp_menu.to_csv("milp_menu_dm_ht_obesity.csv", index=False)
milp_evaluation.to_csv("milp_menu_evaluation_dm_ht_obesity.csv", index=False)

milp_totals.reset_index().rename(
    columns={"index": "nutrient", 0: "total"}
).to_csv("milp_menu_nutrient_total_dm_ht_obesity.csv", index=False)

print("Saved:")
print("- milp_menu_dm_ht_obesity.csv")
print("- milp_menu_evaluation_dm_ht_obesity.csv")
print("- milp_menu_nutrient_total_dm_ht_obesity.csv")

Saved:
- milp_menu_dm_ht_obesity.csv
- milp_menu_evaluation_dm_ht_obesity.csv
- milp_menu_nutrient_total_dm_ht_obesity.csv
